<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


**Research disclosure — AI-assisted coursework.** Prepared for Amir Saulebekov. Outputs below are generated by executing this notebook, not copied from another learner.

Uses the real Wikipedia revision requested by the lab, downloaded on 21 October 2026. Rows with nonnumeric flight identifiers (including Falcon Heavy identifiers) are excluded by the lab rule. Annotation text is removed; payload mass strings are retained rather than inventing numbers for classified payloads.

# **Space X  Falcon 9 First Stage Landing Prediction**


## Web scraping Falcon 9 and Falcon Heavy Launches Records from Wikipedia


Estimated time needed: **40** minutes


In this lab, you will be performing web scraping to collect Falcon 9 historical launch records from a Wikipedia page titled `List of Falcon 9 and Falcon Heavy launches`

https://en.wikipedia.org/wiki/List_of_Falcon_9_and_Falcon_Heavy_launches


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_1_L2/images/Falcon9_rocket_family.svg)


Falcon 9 first stage will land successfully


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/api/Images/landing_1.gif)


Several examples of an unsuccessful landing are shown here:


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/api/Images/crash.gif)


More specifically, the launch records are stored in a HTML table shown below:


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_1_L2/images/falcon9-launches-wiki.png)


  ## Objectives
Web scrap Falcon 9 launch records with `BeautifulSoup`: 
- Extract a Falcon 9 launch records HTML table from Wikipedia
- Parse the table and convert it into a Pandas data frame


First let's import required packages for this lab


In [1]:
# Dependencies are already installed in the project environment; no runtime installation.

In [2]:
import sys

import requests
from bs4 import BeautifulSoup
import re
import unicodedata
import pandas as pd

and we will provide some helper functions for you to process web scraped HTML table


In [3]:
def date_time(table_cells):
    """
    This function returns the data and time from the HTML  table cell
    Input: the  element of a table data cell extracts extra row
    """
    return [data_time.strip() for data_time in list(table_cells.strings)][0:2]

def booster_version(table_cells):
    """
    This function returns the booster version from the HTML  table cell 
    Input: the  element of a table data cell extracts extra row
    """
    out=''.join([booster_version for i,booster_version in enumerate( table_cells.strings) if i%2==0][0:-1])
    return out

def landing_status(table_cells):
    """
    This function returns the landing status from the HTML table cell 
    Input: the  element of a table data cell extracts extra row
    """
    out=[i for i in table_cells.strings][0]
    return out


def get_mass(table_cells):
    mass=unicodedata.normalize("NFKD", table_cells.text).strip()
    if mass:
        mass.find("kg")
        new_mass=mass[0:mass.find("kg")+2]
    else:
        new_mass=0
    return new_mass


def extract_column_from_header(row):
    """
    This function returns the landing status from the HTML table cell 
    Input: the  element of a table data cell extracts extra row
    """
    if (row.br):
        row.br.extract()
    if row.a:
        row.a.extract()
    if row.sup:
        row.sup.extract()
        
    colunm_name = ' '.join(row.contents)
    
    # Filter the digit and empty names
    if not(colunm_name.strip().isdigit()):
        colunm_name = colunm_name.strip()
        return colunm_name    


To keep the lab tasks consistent, you will be asked to scrape the data from a snapshot of the  `List of Falcon 9 and Falcon Heavy launches` Wikipage updated on
`9th June 2021`


In [4]:
static_url = "https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/91.0.4472.124 Safari/537.36"
}

Next, request the HTML page from the above URL and get a `response` object


### TASK 1: Request the Falcon9 Launch Wiki page from its URL


First, let's perform an HTTP GET method to request the Falcon9 Launch HTML page, as an HTTP response.


In [5]:
from pathlib import Path
snapshot_path = Path('../wikipedia_snapshot.html')
html = snapshot_path.read_text(encoding='utf-8')
print('Public historical snapshot:', static_url)
print('Downloaded HTML bytes:', snapshot_path.stat().st_size)

Public historical snapshot: https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922
Downloaded HTML bytes: 3293320


Create a `BeautifulSoup` object from the HTML `response`


In [6]:
soup = BeautifulSoup(html, 'html.parser')

Print the page title to verify if the `BeautifulSoup` object was created properly 


In [7]:
print(soup.title.get_text())

List of Falcon 9 and Falcon Heavy launches - Wikipedia


### TASK 2: Extract all column/variable names from the HTML table header


Next, we want to collect all relevant column names from the HTML table header


Let's try to find all tables on the wiki page first. If you need to refresh your memory about `BeautifulSoup`, please check the external reference link towards the end of this lab


In [8]:
html_tables = soup.find_all('table')
print('Tables:', len(html_tables))

Tables: 29


Starting from the third table is our target table contains the actual launch records.


In [9]:
first_launch_table = html_tables[2]
print(first_launch_table.get_text(' ', strip=True)[:700])

Flight No. Date and time ( UTC ) Version, Booster [ b ] Launch site Payload [ c ] Payload mass Orbit Customer Launch outcome Booster landing 1 4 June 2010, 18:45 F9 v1.0 [ 7 ] B0003.1 [ 8 ] CCAFS , SLC-40 Dragon Spacecraft Qualification Unit LEO SpaceX Success Failure [ 9 ] [ 10 ] (parachute) First flight of Falcon 9 v1.0. [ 11 ] Used a boilerplate version of Dragon capsule which was not designed to separate from the second stage. ( more details below ) Attempted to recover the first stage by parachuting it into the ocean, but it burned up on reentry, before the parachutes even deployed. [ 12 ] 2 8 December 2010, 15:43 [ 13 ] F9 v1.0 [ 7 ] B0004.1 [ 8 ] CCAFS , SLC-40 Dragon demo flight C1 (


You should able to see the columns names embedded in the table header elements `<th>` as follows:


```
<tr>
<th scope="col">Flight No.
</th>
<th scope="col">Date and<br/>time (<a href="/wiki/Coordinated_Universal_Time" title="Coordinated Universal Time">UTC</a>)
</th>
<th scope="col"><a href="/wiki/List_of_Falcon_9_first-stage_boosters" title="List of Falcon 9 first-stage boosters">Version,<br/>Booster</a> <sup class="reference" id="cite_ref-booster_11-0"><a href="#cite_note-booster-11">[b]</a></sup>
</th>
<th scope="col">Launch site
</th>
<th scope="col">Payload<sup class="reference" id="cite_ref-Dragon_12-0"><a href="#cite_note-Dragon-12">[c]</a></sup>
</th>
<th scope="col">Payload mass
</th>
<th scope="col">Orbit
</th>
<th scope="col">Customer
</th>
<th scope="col">Launch<br/>outcome
</th>
<th scope="col"><a href="/wiki/Falcon_9_first-stage_landing_tests" title="Falcon 9 first-stage landing tests">Booster<br/>landing</a>
</th></tr>
```


Next, we just need to iterate through the `<th>` elements and apply the provided `extract_column_from_header()` to extract column name one by one


In [10]:
column_names = []
for th in first_launch_table.find('tr').find_all('th', recursive=False):
    for annotation in th.select('sup.reference'):
        annotation.decompose()
    name = th.get_text(' ', strip=True)
    if name and not name.isdigit():
        column_names.append(name)

Check the extracted column names


In [11]:
print(column_names)

['Flight No.', 'Date and time ( UTC )', 'Version, Booster', 'Launch site', 'Payload', 'Payload mass', 'Orbit', 'Customer', 'Launch outcome', 'Booster landing']


## TASK 3: Create a data frame by parsing the launch HTML tables


We will create an empty dictionary with keys from the extracted column names in the previous task. Later, this dictionary will be converted into a Pandas dataframe


In [12]:
columns = ['Flight No.', 'Date', 'Time', 'Version Booster', 'Launch site',
           'Payload', 'Payload mass', 'Orbit', 'Customer', 'Launch outcome', 'Booster landing']
launch_dict = {name: [] for name in columns}

Next, we just need to fill up the `launch_dict` with launch records extracted from table rows.


Usually, HTML tables in Wiki pages are likely to contain unexpected annotations and other types of noises, such as reference links `B0004.1[8]`, missing values `N/A [e]`, inconsistent formatting, etc.


To simplify the parsing process, we have provided an incomplete code snippet below to help you to fill up the `launch_dict`. Please complete the following code snippet with TODOs or you can choose to write your own logic to parse all launch tables:


In [13]:
def clean(cell):
    clone = BeautifulSoup(str(cell), 'html.parser')
    for annotation in clone.select('sup.reference'):
        annotation.decompose()
    return ' '.join(clone.stripped_strings)

skipped = []
for table in soup.select('table.wikitable.plainrowheaders.collapsible'):
    for row in table.find_all('tr', recursive=False) + table.select('tbody > tr'):
        th = row.find('th', recursive=False)
        if th is None:
            continue
        flight = clean(th)
        if not flight.isdigit():
            continue
        cells = row.find_all('td', recursive=False)
        if len(cells) != 9:
            skipped.append({'flight': flight, 'cells':len(cells)})
            continue
        text = [clean(c) for c in cells]
        match = re.match(r'(.+?\d{4}),?\s+(\d{2}:\d{2}(?::\d{2})?)', text[0])
        date, time = match.groups() if match else (text[0], '')
        values = [int(flight), date, time] + text[1:]
        for key, value in zip(columns, values):
            launch_dict[key].append(value)
print('Parsed rows:', len(launch_dict['Flight No.']), 'Skipped nonstandard rows:', skipped)

Parsed rows: 121 Skipped nonstandard rows: []


After you have fill in the parsed launch record values into `launch_dict`, you can create a dataframe from it.


In [14]:
df = pd.DataFrame(launch_dict).drop_duplicates(subset=['Flight No.'])
assert df['Flight No.'].is_unique and len(df) > 100
display(df.head())
display(df.tail())
print('Missing values by field:')
display(df.isna().sum())
df.to_csv('spacex_web_scraped.csv', index=False)

,Flight No.,Date,Time,Version Booster,Launch site,Payload,Payload mass,Orbit,Customer,Launch outcome,Booster landing
0,1,4 June 2010,18:45,F9 v1.0 B0003.1,"CCAFS , SLC-40",Dragon Spacecraft Qualification Unit,,LEO,SpaceX,Success,Failure (parachute)
1,2,8 December 2010,15:43,F9 v1.0 B0004.1,"CCAFS , SLC-40",Dragon demo flight C1 (Dragon C101),,LEO ( ISS ),NASA ( COTS ) NRO,Success,Failure (parachute)
2,3,22 May 2012,07:44,F9 v1.0 B0005.1,"CCAFS , SLC-40",Dragon demo flight C2+ (Dragon C102),"525 kg (1,157 lb)",LEO ( ISS ),NASA ( COTS ),Success,No attempt
3,4,8 October 2012,00:35,F9 v1.0 B0006.1,"CCAFS , SLC-40",SpaceX CRS-1 (Dragon C103),"4,700 kg (10,400 lb)",LEO ( ISS ),NASA ( CRS ),Success,No attempt
4,5,1 March 2013,15:10,F9 v1.0 B0007.1,"CCAFS , SLC-40",SpaceX CRS-2 (Dragon C104),"4,877 kg (10,752 lb)",LEO ( ISS ),NASA ( CRS ),Success,No attempt


,Flight No.,Date,Time,Version Booster,Launch site,Payload,Payload mass,Orbit,Customer,Launch outcome,Booster landing
116,117,9 May 2021,06:42,F9 B5 ♺ B1051.10,"CCSFS , SLC-40",Starlink 27 v1.0 (60 satellites),"15,600 kg (34,400 lb)",LEO,SpaceX,Success,Success (drone ship)
117,118,15 May 2021,22:56,F9 B5 ♺ B1058.8,"KSC , LC-39A",Starlink 26 v1.0 (52 Satellites) Capella -6 & ...,"~ 14,000 kg (31,000 lb)",LEO,SpaceX Capella Space and Tyvak,Success,Success (drone ship)
118,119,26 May 2021,18:59,F9 B5 ♺ B1063.2,"CCSFS , SLC-40",Starlink 28 v1.0 (60 Satellites),"15,600 kg (34,400 lb)",LEO,SpaceX,Success,Success (drone ship)
119,120,3 June 2021,17:29,F9 B5 B1067.1,"KSC , LC-39A",SpaceX CRS-22 ( Dragon C209.1 ),"3,328 kg (7,337 lb)",LEO ( ISS ),NASA ( CRS ),Success,Success (drone ship)
120,121,6 June 2021,04:26,F9 B5 ♺ B1061.3,"CCSFS , SLC-40",SXM-8,"7,000 kg (15,000 lb)",GTO,Sirius XM,Success,Success (drone ship)


Missing values by field:


Flight No.         0
Date               0
Time               0
Version Booster    0
Launch site        0
Payload            0
Payload mass       0
Orbit              0
Customer           0
Launch outcome     0
Booster landing    0
dtype: int64

We can now export it to a <b>CSV</b> for the next section, but to make the answers consistent and in case you have difficulties finishing this lab. 

Following labs will be using a provided dataset to make each lab independent. 


<code>df.to_csv('spacex_web_scraped.csv', index=False)</code>


## Authors


<a href="https://www.linkedin.com/in/yan-luo-96288783/">Yan Luo</a>


<a href="https://www.linkedin.com/in/nayefaboutayoun/">Nayef Abou Tayoun</a>


<!--
## Change Log
-->


<!--
| Date (YYYY-MM-DD) | Version | Changed By | Change Description      |
| ----------------- | ------- | ---------- | ----------------------- |
| 2021-06-09        | 1.0     | Yan Luo    | Tasks updates           |
| 2020-11-10        | 1.0     | Nayef      | Created the initial version |
-->


Copyright © 2021 IBM Corporation. All rights reserved.
